In [1]:
!nvidia-smi

Tue Apr 28 08:22:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%bash
cd ..
rm -rf ruleformer-findkg/

In [2]:
!git clone https://github.com/rud-rax/ruleformer-findkg.git

Cloning into 'ruleformer-findkg'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 40 (delta 9), reused 38 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 4.60 MiB | 8.96 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [7]:
%cd ruleformer-findkg/

[Errno 2] No such file or directory: 'ruleformer-findkg/'
/content/ruleformer-findkg


In [4]:
!pwd

/content/ruleformer-findkg


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
import os

GDRIVE_HOME = '/content/drive/MyDrive/298-pg'
DATASET_PATH = os.path.join(GDRIVE_HOME , "pgdataset")
EXPS_PATH = os.path.join(GDRIVE_HOME , "experiments")

In [ ]:
# Train Module to extract rules

!python translate.py -data=$DATASET_PATH -jump=2 -padding=100 -batch_size=16 -desc=aa -exps=$EXPS_PATH -savestep=5

save at:/content/drive/MyDrive/298-pg/experiments/aa-j2_20260428_08:43:54
/content/ruleformer-findkg/transformer/dataset.py:35: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  self.relations = [torch.sparse.FloatTensor(indices[i], values[i], size).coalesce() for i in range(self.pos_rels)]
         train-1  MRR:0.10056 @1:0.03319 @3:0.12908 @10:0.24868 LOS:1.02679 Time:17.3secs     
         train-2  MRR:0.10382 @1:0.03372 @3:0.13224 @10:0.25132 LOS:1.03031 Time:34.5secs     
         train-3  MRR:0.10629 @1:0.03214 @3:0.14067 @10:0.25395 LOS:1.02559 Time:52.3secs     
         train-4  MRR:0.10802 @1:0.03372 @3:0.14278 @10:0.26027 LOS:1.02412 Time:70.5secs     
         train-5  MRR:0.10768 @1:0.03477 @3:0.14015 @10:0.25342 LOS:1.02487 Time:89.6secs     
         valid-5  MRR:0.06302 @1:0.01835 

In [36]:
CKPT_PATH = os.path.join(EXPS_PATH , "aa-j2_20260428_08:43:54" , "Translator10.ckpt")
DESC_PATH = os.path.join(EXPS_PATH , "aa-j2_20260428_08:43:54")

In [35]:
DESC_PATH

'/content/drive/MyDrive/298-pg/experiments/aa-j2_20260428_08:43:54'

In [ ]:
# Rule Mining

!python translate.py \
  -data=$DATASET_PATH \
  -jump=2 \
  -padding=100 \
  -batch_size=16 \
  -desc=$DESC_PATH \
  -ckpt=$CKPT_PATH  \
  -decode_rule \
  -the_rel=0.3 \
  -the_rel_min=0.1 \
  -the_all=0.05

save at:/content/drive/MyDrive/298-pg/experiments/aa-j2_20260428_08:43:54-j2_20260428_09:19:28
/content/ruleformer-findkg/transformer/dataset.py:35: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  self.relations = [torch.sparse.FloatTensor(indices[i], values[i], size).coalesce() for i in range(self.pos_rels)]
Write 889-124 Rule(s)[Info] Finished.
Writing Rules...


In [41]:
!pwd

/content/ruleformer-findkg


In [42]:
%cd ..

/content


In [44]:
!ls

apply_rules.py	drive  ruleformer-findkg  sample_data


In [45]:
RULES_PATH = "/content/drive/MyDrive/298-pg/experiments/aa-j2_20260428_08:43:54-j2_20260428_09:19:28/rules.txt"
TRAIN_PATH = "/content/drive/MyDrive/298-pg/pgdataset/train.txt"

In [ ]:
# Apply rules to predict triplets

!python -m apply_rules \
  --rules_file $RULES_PATH \
  --ruleformer_train $TRAIN_PATH \
  --data_dir FinDKG_dataset \
  --dataset FinDKG \
  --output sym_triplets.tsv \
  --min_weight 0.1 \
  --min_count 1


[apply_rules] Loading training KG: /content/drive/MyDrive/298-pg/pgdataset/train.txt
  Training triples: 949
[apply_rules] Parsing rules: /content/drive/MyDrive/298-pg/experiments/aa-j2_20260428_08:43:54-j2_20260428_09:19:28/rules.txt
  Rules passing filters (weight≥0.1, count≥1): 125
  [1.000|2] inv_Relate_To <- inv_Relate_To·inv_Negative_Impact_On  +0
  [1.000|3] inv_Raise <- <slf>·<slf>  +0
  [1.000|1] inv_Raise <- <slf>·inv_Invests_In  +0
  [1.000|1] inv_Control <- <slf>·<slf>  +0
  [1.000|1] inv_Control <- inv_Relate_To·inv_Has  +0
  [1.000|3] inv_Control <- inv_Positive_Impact_On·inv_Introduce  +0
  [1.000|4] inv_Invests_In <- <slf>·<slf>  +0
  [1.000|2] inv_Invests_In <- <slf>·inv_Negative_Impact_On  +0
  [1.000|1] Is_Member_Of <- <slf>·Is_Member_Of  +0
  [1.000|4] Relate_To <- Is_Member_Of·Is_Member_Of  +2
  [1.000|1] Relate_To <- Introduce·inv_Introduce  +0
  [1.000|5] Relate_To <- inv_Control·inv_Relate_To  +0
  [1.000|1] Relate_To <- inv_Relate_To·Is_Member_Of  +0
  [1.000|5

In [51]:
!mkdir -p FinDKG_dataset/FinDKG

In [52]:
!mv entity2id.txt FinDKG_dataset/FinDKG/
!mv relation2id.txt FinDKG_dataset/FinDKG/
